# 06 — H7 Extended-Training Results

This CPU-only notebook presents the H7 continuation experiment from the versioned results snapshot. It does not load checkpoints or datasets, and it never runs training, sampling, or metric computation.

H7 asks a different question from the CA16 architecture ablation: holding the selected CA8+CA16 architecture and the frozen training protocol fixed, what changes when the three R10 runs are continued from 20k to 40k successful optimizer steps?

In [ ]:
from __future__ import annotations

import json
import statistics
from pathlib import Path

import matplotlib.pyplot as plt


def find_repository_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'docs/experiments/data/h7_results_v1.json').is_file():
            return candidate
    raise FileNotFoundError('Could not locate docs/experiments/data/h7_results_v1.json')


ROOT = find_repository_root(Path.cwd())
SNAPSHOT_PATH = ROOT / 'docs/experiments/data/h7_results_v1.json'
snapshot = json.loads(SNAPSHOT_PATH.read_text(encoding='utf-8'))
assert snapshot['schema'] == 'h7_results_snapshot_v1'
rows = sorted(snapshot['rows'], key=lambda row: row['seed'])
assert [row['seed'] for row in rows] == [2026, 2027, 2028]

COLORS = {2026: '#31688e', 2027: '#35b779', 2028: '#440154'}
MARKERS = {2026: 'o', 2027: 's', 2028: '^'}

print(f'Repository root: {ROOT}')
print(f'Snapshot: {SNAPSHOT_PATH.relative_to(ROOT)}')
print('Seeds:', [row['seed'] for row in rows])

## Experimental design

Each seed (2026, 2027, 2028) continues a frozen CA8+CA16 checkpoint from 20,000 to 40,000 successful optimizer steps. The continuation retains the optimizer, AMP, sampler/RNG, and EMA state. R11 evaluates semantic adherence on the frozen ID_VAL and COMP_VAL prompts.

R13 uses the corrected historical batch-8 generation contract: per-lane RNG semantics, final-batch padding, EMA denoiser and text encoder, FP32 ancestral DDPM sampling with T=1000, and the canonical 804-image pool (466 ID and 338 COMP). KID is the primary distributional guardrail; FID is secondary and complementary.

In [ ]:
print('R11 protocol:', snapshot['protocols']['r11_tag'], snapshot['protocols']['r11_commit'])
print('R13 protocol:', snapshot['protocols']['r13_tag'], snapshot['protocols']['r13_commit'])
for row in rows:
    terminal = row['terminal']
    pool = row['pool']
    assert terminal['seed'] == row['seed'] and terminal['step'] == 40000
    assert pool['status'] == 'COMPLETE' and pool['image_count'] == 804
    print(f"seed {row['seed']}: terminal {terminal['sha256'][:12]}..., pool {pool['image_count']} images")

## R11 semantic adherence

The primary endpoint is COMP_VAL equal-family macro MAA. COMP_VAL MacroPSR and ID_VAL MAA are secondary semantic endpoints. The following table and summaries are calculated directly from the snapshot; changes are paired within seed (40k minus 20k).

In [ ]:
SEMANTIC = {'comp_maa': 'COMP_VAL macro MAA', 'comp_psr': 'COMP_VAL MacroPSR', 'id_maa': 'ID_VAL MAA'}

def paired_values(key: str):
    before = [row['metrics_20k'][key] for row in rows]
    after = [row['metrics_40k'][key] for row in rows]
    return before, after, [new - old for old, new in zip(before, after)]

for key, label in SEMANTIC.items():
    before, after, delta = paired_values(key)
    print(f'\n{label}')
    for row, old, new, change in zip(rows, before, after, delta):
        print(f"  seed {row['seed']}: {old:.6f} -> {new:.6f}  (delta {change:+.6f})")
    print(f"  mean paired delta: {statistics.mean(delta):+.6f}; sample SD: {statistics.stdev(delta):.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2), constrained_layout=True)
for row in rows:
    y = [row['metrics_20k']['comp_maa'], row['metrics_40k']['comp_maa']]
    seed = row['seed']
    ax.plot([0, 1], y, color=COLORS[seed], marker=MARKERS[seed], label=f'seed {seed}')
ax.set(xlim=(-0.15, 1.15), ylim=(0, 1), xticks=[0, 1], xticklabels=['20k', '40k'],
       xlabel='Successful optimizer steps', ylabel='Macro MAA',
       title='COMP_VAL equal-family macro MAA (higher is better)')
ax.grid(axis='y', alpha=0.25)
ax.legend(title='Paired seed')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2), constrained_layout=True)
for row in rows:
    y = [row['metrics_20k']['comp_psr'], row['metrics_40k']['comp_psr']]
    seed = row['seed']
    ax.plot([0, 1], y, color=COLORS[seed], marker=MARKERS[seed], label=f'seed {seed}')
ax.set(xlim=(-0.15, 1.15), ylim=(0, 1), xticks=[0, 1], xticklabels=['20k', '40k'],
       xlabel='Successful optimizer steps', ylabel='MacroPSR',
       title='COMP_VAL MacroPSR (higher is better)')
ax.grid(axis='y', alpha=0.25)
ax.legend(title='Paired seed')
plt.show()

## R13 distributional quality

KID and FID are calculated from the same validated R13 pool for ID, COMP, and GLOBAL populations. Both are lower-is-better quantities. KID uses the frozen 100-subset procedure; FID is a secondary metric and is especially sensitive to the modest generated-sample sizes.

In [ ]:
for prefix in ('kid', 'fid'):
    print(f'\n{prefix.upper()} (lower is better)')
    for role in ('id', 'comp', 'global'):
        key = f'{prefix}_{role}'
        _, _, delta = paired_values(key)
        values = ', '.join(
            f"{row['seed']}: {row['metrics_20k'][key]:.6f} -> {row['metrics_40k'][key]:.6f}"
            for row in rows
        )
        print(f'  {role.upper()}: {values}; mean paired delta {statistics.mean(delta):+.6f}; sample SD {statistics.stdev(delta):.6f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.8), sharey=True, constrained_layout=True)
for ax, role in zip(axes, ('id', 'comp', 'global')):
    key = f'kid_{role}'
    for row in rows:
        seed = row['seed']
        ax.plot([0, 1], [row['metrics_20k'][key], row['metrics_40k'][key]],
                color=COLORS[seed], marker=MARKERS[seed], label=f'seed {seed}')
    ax.set(title=role.upper(), xlim=(-0.15, 1.15), ylim=(0, None), xticks=[0, 1], xticklabels=['20k', '40k'])
    ax.grid(axis='y', alpha=0.25)
axes[0].set_ylabel('KID (lower is better)')
axes[0].legend(title='Paired seed')
fig.suptitle('R13 KID by population')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.8), sharey=True, constrained_layout=True)
for ax, role in zip(axes, ('id', 'comp', 'global')):
    key = f'fid_{role}'
    for row in rows:
        seed = row['seed']
        ax.plot([0, 1], [row['metrics_20k'][key], row['metrics_40k'][key]],
                color=COLORS[seed], marker=MARKERS[seed], label=f'seed {seed}')
    ax.set(title=role.upper(), xlim=(-0.15, 1.15), ylim=(0, None), xticks=[0, 1], xticklabels=['20k', '40k'])
    ax.grid(axis='y', alpha=0.25)
axes[0].set_ylabel('FID (secondary; lower is better)')
axes[0].legend(title='Paired seed')
fig.suptitle('R13 FID by population')
plt.show()

## Compositional diagnostics

CV1–CV4 are verified 40k COMP_VAL family diagnostics. No authoritative persisted 20k family breakdown is available, so this section deliberately does not present them as 20k–40k paired comparisons.

In [ ]:
families = ('CV1', 'CV2', 'CV3', 'CV4')
fig, ax = plt.subplots(figsize=(7, 4.2), constrained_layout=True)
for row in rows:
    seed = row['seed']
    values = [row['diagnostics_40k']['family_maa'][family] for family in families]
    ax.plot(families, values, color=COLORS[seed], marker=MARKERS[seed], label=f'seed {seed}')
    print(f"seed {seed}: " + ', '.join(f'{family}={value:.6f}' for family, value in zip(families, values)))
ax.set(ylim=(0, 1), ylabel='40k MAA (higher is better)',
       title='COMP_VAL family diagnostics (40k only)')
ax.grid(axis='y', alpha=0.25)
ax.legend(title='Seed')
plt.show()

## Joint interpretation

The next cell reports direction consistency from the snapshot rather than asserting a significance result. Semantic adherence and the two distributional measures address different properties: their jointly favourable directions are descriptive evidence, not perceptual-quality proof or independent held-out confirmation.

In [ ]:
for key, label, favourable in [
    ('comp_maa', 'COMP_VAL macro MAA', 'higher'),
    ('comp_psr', 'COMP_VAL MacroPSR', 'higher'),
    ('id_maa', 'ID_VAL MAA', 'higher'),
    ('kid_global', 'GLOBAL KID', 'lower'),
    ('fid_global', 'GLOBAL FID', 'lower'),
]:
    _, _, delta = paired_values(key)
    consistent = all(change > 0 for change in delta) if favourable == 'higher' else all(change < 0 for change in delta)
    print(f"{label}: direction favourable for all seeds = {consistent}; paired changes = {[round(change, 6) for change in delta]}")

## Threats to validity and conclusion

COMP_VAL was reused in historical checkpoint selection, so it is not an independent held-out confirmation. There are only three training seeds. Historical 20k ID_VAL PSR and CV1–CV4 breakdowns have no authoritative persisted source and are absent from paired claims. KID and FID are distributional proxies, not direct perceptual ratings or diversity guarantees; FID is secondary and small-sample sensitive.

The R13 batch-8 correction was validated for algorithmic compatibility with the historical sampler; an interrupted batch-1 pool was isolated and not used. The evidence therefore supports a carefully scoped paired continuation result, while leaving independent confirmation and broader generalization unresolved.

## Evidence and reproduction

This notebook reads [`docs/experiments/data/h7_results_v1.json`](../docs/experiments/data/h7_results_v1.json). The companion [H7 report](../docs/experiments/h7_extended_training.md) and [`scripts/generate_h7_report.py`](../scripts/generate_h7_report.py) provide the versioned prose and CPU-only figure generator.

Requirements: Python, the standard library, and Matplotlib. No CUDA, checkpoints, dataset, external CA16/H7 result directory, or network access is required for the snapshot-only analysis. Strict verification against external frozen artifacts is documented in `REPRODUCIBILITY.md`.